# Запуск приложения «Ассистент по лекциям» в Colab
Streamlit (интерфейс) + FastAPI (сервер с моделями).

1. *Среда выполнения → Сменить среду выполнения → T4 GPU*.
2. Запускайте ячейки по порядку. Последняя выдаст ссылку на приложение.

## 1. Код из GitHub

In [ ]:
REPO = "https://github.com/ageraforreal/lecture-assistant.git"
!git clone -q {REPO}
%cd lecture-assistant

## 2. Библиотеки

In [ ]:
!pip -q install -r requirements.txt

## 3. Тесты

In [ ]:
!python -m pytest -v

## 4. Запуск сервера FastAPI и интерфейса Streamlit
Модели загружаются при первом запросе (1–3 минуты).

In [ ]:
!nohup uvicorn api:app --port 8000 > api.log 2>&1 &
!nohup streamlit run app.py --server.port 8501 --server.headless true > ui.log 2>&1 &
!sleep 5 && curl -s localhost:8000/health

## 5. Публичная ссылка на приложение
Откройте ссылку `https://...trycloudflare.com`. Ячейка остаётся запущенной, пока работает приложение.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import re, subprocess
p = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                     stderr=subprocess.PIPE, text=True)
for line in p.stderr:
    m = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if m:
        print("Приложение:", m.group())
        break

## Если что-то не работает
Логи сервера и интерфейса:

In [ ]:
!tail -n 30 api.log ui.log